In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

工具的应用案例
4.1 案例1：使用args_schema
工具的定义与模拟调用：

In [ ]:
from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain.messages import HumanMessage
from langchain_core.utils.function_calling import convert_to_openai_tool


class WeatherSchema(BaseModel):
    city: str = Field(default="北京", description="城市名称")
    if_forecast: bool = Field(default=False, description="是否包含明日天气预报")


@tool("get_weather_and_forecast", description="查询当日天气，可以包含明日天气预报", args_schema=WeatherSchema)
def get_weather(city: str, if_forecast: bool):
    res = f"{city} 今天天气不错"
    if if_forecast:
        res += "\n明天也不错"
    return res


print(convert_to_openai_tool(get_weather))

model_with_tools = model.bind_tools([get_weather])

messages = [HumanMessage("今天杭州天气如何？明天呢？")]

response = model_with_tools.invoke(messages)

messages.append(response)

tool_calls = response.tool_calls

for tool_call in tool_calls:
    if tool_call["name"] == "get_weather_and_forecast":
        tool_msg = get_weather.invoke(tool_call)
        messages.append(tool_msg)
final_response = model_with_tools.invoke(messages)
messages.append(final_response)
for msg in messages:
    msg.pretty_print()

案例3：多工具调用
大模型调用工具是单次推理，即每次运行调用一个工具，当调用多个工具时，需要用户自己管理多次调
用循环。

In [12]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from dotenv import load_dotenv
from rich import print as rprint
from langchain.chat_models import init_chat_model
import os


# 1.定义工具
# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """
        获取指定公司的股票价格信息

    Args:
        company: 公司名称（如：苹果公司, 微软公司, 谷歌公司）
        timeframe: 时间范围（today-今日, week-本周, month-本月）
    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }
    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格: {price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"


# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """
    搜索指定公司的财经新闻

    Args:
        company: 公司名称
    """
    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
            "苹果发布新款iPhone，股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
            "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ]
    }
    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)


def invokeModel():
    # 从.env文件中加载环境变量
    load_dotenv(override=True)
    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    DEEPSEEK_BASE_URL = "https://api.deepseek.com"
    model = init_chat_model(
        model="deepseek-v4-flash",
        model_provider="deepseek",
        api_key=DEEPSEEK_API_KEY,
        base_url=DEEPSEEK_BASE_URL
    )

    # rprint(convert_to_openai_tool(search_news))

    # 2.初始化模型并绑定工具
    tools = [get_stock_price, search_news]
    model_with_tools = model.bind_tools(tools)

    message_list = []
    human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
    # human_message = HumanMessage(content="比较一下微软和苹果的股价")
    # human_message = HumanMessage(content="腾讯最近有什么重大新闻？")
    # human_message = HumanMessage(content="海水为什么是咸的？")
    message_list.append(human_message)

    # 3.工具调用
    while True:
        response = model_with_tools.invoke(message_list)
        message_list.append(response)
        # 如果模型不需要调用工具，直接退出循环
        if not response.tool_calls:
            print("没有工具调用，直接返回答案")
            break
        # 如果有调用工具，处理工具调用响应
        # 4.开发者根据模型的响应，调用工具并获取结果
        for tool_call in response.tool_calls:
            if tool_call["name"] == "get_stock_price":
                stock_result = get_stock_price.invoke(tool_call)
                print("stock_result", stock_result)
                message_list.append(stock_result)
            if tool_call["name"] == "search_news":
                news_result = search_news.invoke(tool_call)
                print("news_result", news_result)
                message_list.append(news_result)
    # print("response", response)
    # print(response.content)
    for msg in message_list:
        msg.pretty_print()


invokeModel()

stock_result content='苹果公司 today价格: 185.2美元' name='get_stock_price' tool_call_id='call_00_GvPn1Vtx7UJ1qXoXl90X8593'
news_result content='苹果发布新款iPhone，股价上涨3%\n苹果与欧盟达成反垄断和解协议\n苹果将在印度扩大生产规模' name='search_news' tool_call_id='call_01_uu0hj2leg9koqAjBMbAI7201'
没有工具调用，直接返回答案
================================ Human Message =================================

苹果公司今天的股价是多少？最近有什么新闻？
================================== Ai Message ==================================

好的，我来同时查询苹果公司的股价和最新新闻！
Tool Calls:
  get_stock_price (call_00_GvPn1Vtx7UJ1qXoXl90X8593)
 Call ID: call_00_GvPn1Vtx7UJ1qXoXl90X8593
  Args:
    company: 苹果公司
    timeframe: today
  search_news (call_01_uu0hj2leg9koqAjBMbAI7201)
 Call ID: call_01_uu0hj2leg9koqAjBMbAI7201
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price

苹果公司 today价格: 185.2美元
================================= Tool Message =================================
Name: search_news

苹果发布新款iPhone，股价上涨3%
苹果与

优化工具定义

In [17]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from dotenv import load_dotenv
from rich import print as rprint
from langchain.chat_models import init_chat_model
import os
from pydantic import BaseModel, Field
from langchain_core.utils.function_calling import convert_to_openai_tool


class StockPriceInput(BaseModel):
    company: str
    timeframe: str = Field(
        default="today",
        description="时间范围（today-今日, week-本周, month-本月）"
    )


class AIPriceInput(BaseModel):
    company: str


# 1.定义工具
# 定义股票查询工具
@tool(name_or_callable="get_stock_price", args_schema=StockPriceInput, description="获取指定公司的股票价格信息")
def get_stock_price(company: str, timeframe: str = "today") -> str:
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }
    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格: {price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"


# 定义新闻搜索工具
@tool(name_or_callable="search_news", description="搜索指定公司的财经新闻", args_schema=AIPriceInput)
def search_news(company: str) -> str:
    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
            "苹果发布新款iPhone，股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
            "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ]
    }
    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)


rprint(convert_to_openai_tool(search_news))
rprint(convert_to_openai_tool(get_stock_price))



{
    'type': 'function',
    'function': {
        'name': 'search_news',
        'description': '搜索指定公司的财经新闻',
        'parameters': {'properties': {'company': {'type': 'string'}}, 'required': ['company'], 'type': 'object'}
    }
}

{
    'type': 'function',
    'function': {
        'name': 'get_stock_price',
        'description': '获取指定公司的股票价格信息',
        'parameters': {
            'properties': {
                'company': {'type': 'string'},
                'timeframe': {
                    'default': 'today',
                    'description': '时间范围（today-今日, week-本周, month-本月）',
                    'type': 'string'
                }
            },
            'required': ['company'],
            'type': 'object'
        }
    }
}